# 02 · ASVspoof 5 — XLSR-MamBo

Primo esperimento allo stato dell'arte sui subset preparati nel notebook 01 (forma d'onda grezza, non le Mel).

**Modello**: XLSR-MamBo, configurazione MamBo-3-Hydra-N3 (Ng et al., *XLSR-MamBo: Scaling the Hybrid
Mamba-Attention Backbone for Audio Deepfake Detection*, arXiv 2601.02944, ACL 2026 Findings;
codice originale https://github.com/saki-ciallo/MamBo-for-ADD, MIT). XLS-R 300M fine-tunato →
5 unità [3×Hydra, MLP, MHA, MLP] → attention pooling → 2 classi. Port in PyTorch puro in `src/thesis/models/`.

**Protocollo**: train sul subset train (5000+5000), scelta dell'epoca **solo** sull'EER del dev
(2500+2500, attacchi A09–A16), test una sola volta sull'eval (5000+5000, attacchi A17–A32 e codec mai visti).
3 seed; metriche EER, minDCF, actDCF, Cllr, EER per attacco e per codec, IC 95% bootstrap dell'EER.

**Differenze dal paper** (dovute alla 3060 da 8 GB / Windows): XLS-R da Hugging Face invece che da torchaudio
(stessi pesi); CNN iniziale di XLS-R congelata e gradient checkpointing; batch 8 × 4 di accumulo (32 effettivo);
focal loss con α = 0.5/0.5 perché i subset sono bilanciati; nessuna RawBoost (esperimento separato);
training sui subset di ASVspoof 5, non su ASVspoof 2019 LA.

Il training gira **fuori da Jupyter** (`python -m thesis.train`), così non si ferma se VS Code si scollega.
Questo notebook lo lancia e poi analizza i risultati salvati su file.

## 0 · Setup

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # il notebook gira da notebooks/
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis import config
from thesis.train import TrainConfig

print(config.describe())

## 1 · Configurazione

In [ ]:
CONFIG = REPO / "configs" / "mambo_asvspoof5.json"
SEEDS = [1, 2, 3]

cfg = TrainConfig.from_json(CONFIG)
RUN_DIR = config.ARTIFACTS_ROOT / "runs" / cfg.name
print(json.dumps(json.loads(CONFIG.read_text()), indent=2))
print("Risultati in:", RUN_DIR)

## 2 · Lancio del training in background

La cella avvia un processo separato e torna subito: il training continua anche chiudendo VS Code.
I seed già completati (con `metrics.json`) vengono saltati, quindi si può rilanciare senza perdere nulla.
Il log testuale è in `RUN_DIR/train.log`. **Lanciare una sola volta**: due training insieme non stanno in 8 GB.

In [ ]:
LAUNCH = False  # metti True per avviare

if LAUNCH:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    log = open(RUN_DIR / "train.log", "a", encoding="utf-8")
    flags = 0
    if sys.platform == "win32":  # processo staccato dal kernel e da VS Code
        flags = subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.CREATE_BREAKAWAY_FROM_JOB
    proc = subprocess.Popen(
        [sys.executable, "-m", "thesis.train", "--config", str(CONFIG), "--seed", *map(str, SEEDS)],
        cwd=REPO, stdout=log, stderr=subprocess.STDOUT, creationflags=flags,
        env={**os.environ, "PYTHONPATH": str(REPO / "src")},
    )
    print("Avviato, PID", proc.pid)

## 3 · Monitoraggio

`stato()` legge `train.log` e i `log.csv` senza toccare il training. Mostra seed ed epoca correnti,
batch fatti/totali, velocità, ETA e l'EER dev delle epoche finite, e segnala in evidenza:
processo di training non attivo, log fermo da più di 10 minuti, loss NaN/inf, errori nel log.

Con `AGGIORNA_MIN > 0` la cella si aggiorna da sola ogni 60 s per quei minuti
(si ferma con il pulsante ■ di interruzione del kernel; il training non ne risente).

In [ ]:
import re
import time
from datetime import datetime

import psutil
from IPython.display import clear_output

AGGIORNA_MIN = 0      # es. 60 = aggiorna ogni minuto per un'ora
LOG_FERMO_MIN = 10

TQDM = re.compile(r"epoca (\d+):\s+\d+%.*?\|\s*(\d+)/(\d+) \[([\d:]+)<([\d:?]+),\s*([\d.?]+)(s/it|it/s).*?loss=([\w.+-]+)")


def processo_attivo() -> bool:
    """C'è un processo `python -m thesis.train` lanciato con questa config?"""
    for p in psutil.process_iter(["cmdline"]):
        cmd = " ".join(p.info["cmdline"] or [])
        if "thesis.train" in cmd and CONFIG.name in cmd:
            return True
    return False


def coda_log(path: Path, n_byte: int = 30_000) -> list[str]:
    with open(path, "rb") as f:
        f.seek(max(0, path.stat().st_size - n_byte))
        testo = f.read().decode("utf-8", errors="replace")
    return [r.strip() for r in re.split(r"[\r\n]+", testo) if r.strip()]


def stato():
    print(f"Aggiornato alle {datetime.now():%H:%M:%S} — run: {RUN_DIR}")
    avvisi = []
    log_file = RUN_DIR / "train.log"
    attivo = processo_attivo()
    completati = [s for s in SEEDS if (RUN_DIR / f"seed{s}" / "metrics.json").exists()]

    # Riepilogo per seed dalle epoche finite (log.csv)
    for s in SEEDS:
        csv = RUN_DIR / f"seed{s}" / "log.csv"
        if (RUN_DIR / f"seed{s}" / "metrics.json").exists():
            r = json.loads((RUN_DIR / f"seed{s}" / "metrics.json").read_text())
            print(f"  seed {s}: COMPLETATO — epoca migliore {r['epoca_migliore']}, "
                  f"EER dev {100 * r['dev']['eer']:.2f}%, EER eval {100 * r['eval']['eer']:.2f}%")
        elif csv.exists():
            log = pd.read_csv(csv)
            migliore = log.loc[log.dev_eer.idxmin()]
            print(f"  seed {s}: {len(log)} epoche finite — EER dev ultima {100 * log.dev_eer.iloc[-1]:.2f}%, "
                  f"migliore {100 * migliore.dev_eer:.2f}% (epoca {int(migliore.epoca)}), "
                  f"{log.sec_totale.mean() / 60:.1f} min/epoca, GPU max {log.gpu_max_gb.max()} GB")
            if log[["train_loss", "dev_loss"]].isna().any().any() or np.isinf(log[["train_loss", "dev_loss"]].to_numpy()).any():
                avvisi.append(f"loss NaN/inf nel log.csv del seed {s}")
        elif (RUN_DIR / f"seed{s}" / "config.json").exists():
            print(f"  seed {s}: in corso, prima epoca non ancora finita")
        else:
            print(f"  seed {s}: non ancora iniziato")

    # Avanzamento dentro l'epoca corrente (ultima barra tqdm nel train.log)
    if log_file.exists():
        righe = coda_log(log_file)
        barre = [m for r in righe if (m := TQDM.search(r))]
        if barre and attivo:
            ep, fatti, tot, trascorso, resto, vel, unita, loss = barre[-1].groups()
            bps = float(vel) if unita == "it/s" else 1 / float(vel)
            corrente = next(s for s in SEEDS if s not in completati)
            print(f"\nIn corso: seed {corrente}, epoca {ep}, batch {fatti}/{tot} ({100 * int(fatti) / int(tot):.0f}%), "
                  f"{bps:.2f} batch/s, loss {loss}, fine epoca tra ~{resto}")
            if loss.lower() in ("nan", "inf", "-inf"):
                avvisi.append(f"loss {loss} nell'epoca {ep}")
        fermo_min = (time.time() - log_file.stat().st_mtime) / 60
        if attivo and fermo_min > LOG_FERMO_MIN:
            avvisi.append(f"train.log non si aggiorna da {fermo_min:.0f} minuti: training bloccato?")
        errori = [r for r in righe if re.search(r"Traceback|Error|ERROR|AssertionError|out of memory", r)]
        if errori:
            avvisi.append("errori nel log:\n      " + "\n      ".join(errori[-5:]))
    else:
        print("\ntrain.log non trovato: il training non è mai stato lanciato da qui.")

    if len(completati) == len(SEEDS):
        print("\nTUTTI I SEED COMPLETATI: vai alla sezione 5.")
    elif not attivo:
        avvisi.append("il processo di training NON è attivo e non tutti i seed sono completati "
                      "(crash, riavvio del PC o mai lanciato): rilancia dalla sezione 2")

    if avvisi:
        print("\n" + "!" * 70)
        for a in avvisi:
            print("  ATTENZIONE:", a)
        print("!" * 70)
        if log_file.exists():
            print("\nUltime righe di train.log:")
            for r in coda_log(log_file)[-8:]:
                print("   ", r[:200])
    else:
        print("\nNessun problema rilevato.")


fine = time.time() + 60 * AGGIORNA_MIN
while True:
    clear_output(wait=True)
    stato()
    if time.time() > fine:
        break
    time.sleep(60)

## 4 · Curve di training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for seed in SEEDS:
    log_path = RUN_DIR / f"seed{seed}" / "log.csv"
    if not log_path.exists():
        continue
    log = pd.read_csv(log_path)
    axes[0].plot(log.epoca, log.train_loss, label=f"train s{seed}")
    axes[0].plot(log.epoca, log.dev_loss, "--", label=f"dev s{seed}")
    axes[1].plot(log.epoca, 100 * log.dev_eer, marker="o", label=f"seed {seed}")
axes[0].set(title="Loss (train: focal, dev: cross-entropy)", xlabel="epoca"); axes[0].legend(fontsize=7)
axes[1].set(title="EER dev (%)", xlabel="epoca"); axes[1].legend()
plt.tight_layout()

## 5 · Risultati

Tabella generata dai `metrics.json` (media ± deviazione standard sui seed).
actDCF e Cllr misurano la calibrazione dei punteggi (logit bonafide − logit spoof, senza calibrazione a posteriori).

In [ ]:
results = {s: json.loads((RUN_DIR / f"seed{s}" / "metrics.json").read_text())
           for s in SEEDS if (RUN_DIR / f"seed{s}" / "metrics.json").exists()}
assert results, "Nessun seed completato"
print("Seed completati:", list(results))

rows = []
for s, r in results.items():
    for split in ("dev", "eval"):
        m = r[split]
        rows.append({"seed": s, "split": split, "epoca": r["epoca_migliore"],
                     "EER %": 100 * m["eer"], "IC95 EER %": f"{100 * m['eer_ci95'][0]:.2f}–{100 * m['eer_ci95'][1]:.2f}",
                     "minDCF": m["min_dcf"], "actDCF": m["act_dcf"], "Cllr": m["cllr"]})
per_seed = pd.DataFrame(rows)
display(per_seed.round(4))

summary = per_seed.groupby("split")[["EER %", "minDCF", "actDCF", "Cllr"]].agg(["mean", "std"])
display(summary.round(4))

### EER per attacco e per codec (eval)

In [ ]:
def by_group(key):
    df = pd.DataFrame({s: r["eval"][key] for s, r in results.items()})
    return (100 * df).assign(media=lambda d: d.mean(axis=1)).round(2)

attacks, codecs = by_group("eer_per_attacco"), by_group("eer_per_codec")
display(attacks.T)
display(codecs.T)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), gridspec_kw={"width_ratios": [2, 1]})
attacks["media"].plot.bar(ax=axes[0], title="EER per attacco — eval (%)")
codecs["media"].plot.bar(ax=axes[1], title="EER per codec — eval ('-' = nessuno) (%)")
plt.tight_layout()

### Distribuzione dei punteggi (eval, primo seed)

In [ ]:
s0 = next(iter(results))
scores = pd.read_csv(RUN_DIR / f"seed{s0}" / "scores_eval.csv")
fig, ax = plt.subplots(figsize=(7, 3.5))
for label, g in scores.groupby("label"):
    ax.hist(g["score"], bins=80, alpha=0.6, density=True, label=label)
ax.axvline(-np.log(0.95 / 0.5), color="k", ls="--", lw=1, label="soglia actDCF")
ax.set(title=f"Punteggi eval — seed {s0}", xlabel="logit bonafide − logit spoof"); ax.legend()